# Optuna Hyperparameter Optimization for CompressionPriceNet

This notebook optimizes the hyperparameters of the CompressionPriceNet architecture using Optuna.

In [ ]:
!uv pip install -q optuna torch numpy pandas scikit-learn tqdm matplotlib
import os

!git clone https://github.com/antonawinkler/slm-pricer.git
%cd slm-pricer
!uv pip install -e .
import sys

sys.path.append(
    os.path.abspath("src")
)  # TODO: analyse why installing the package does not suffice on Colab
from google.colab import drive

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from slm_pricer.data import PriceDataset, load_data_from_hf
from slm_pricer.models import CompressionPriceNet
from slm_pricer.training import evaluate_model, train_model
from slm_pricer.utils import (
    convert_back_y,
    transformed_prices,
    create_early_stopping_fn,
)

In [ ]:
@dataclass
class Config:
    """Configuration for Optuna hyperparameter optimization."""

    # Data
    data_percent: int = 10  # Percentage of data to use (1-100)

    # Data preprocessing
    y_transform: str = "log"  # Transform for target prices: "log", "unit", or None

    # Training hyperparameters
    batch_size: int = 64  # Fixed large enough value
    epochs: int = 100
    grad_clip: float = 1.0

    # Caching
    cache_path: Path = Path("/content/drive/MyDrive/Pricer_Embeddings")
    cache_filename_root: str = "llama_fine_tuned_"

    # Optuna settings
    n_trials: int = 50  # Number of hyperparameter trials

    # Reproducibility
    seed: int = 42


CONFIG = Config()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(CONFIG.seed)
np.random.seed(CONFIG.seed)

In [ ]:
print("Loading target prices...")
df_train = load_data_from_hf(split="train", percent=CONFIG.data_percent)
df_val = load_data_from_hf(split="val")
df_test = load_data_from_hf(split="test")

print(f"Loaded {len(df_train)} train, {len(df_val)} val, {len(df_test)} test samples")

## Load Pre-computed Embeddings

In [ ]:
drive.mount("/content/drive")

cache_path = CONFIG.cache_path

X_train = np.array(
    np.load(cache_path / f"{CONFIG.cache_filename_root}train.npy", mmap_mode="r")[
        : len(df_train)
    ]
)

X_val = np.load(cache_path / f"{CONFIG.cache_filename_root}val.npy")
X_test = np.load(cache_path / f"{CONFIG.cache_filename_root}test.npy")


print(f"  Train: {len(X_train)} samples ({X_train.shape})")
print(f"  Val: {len(X_val)} samples ({X_val.shape})")
print(f"  Test: {len(X_test)} samples ({X_test.shape})")

In [ ]:
y_train = transformed_prices(
    df_train["completion"].to_numpy(dtype="float32"), CONFIG.y_transform
)
y_val = transformed_prices(
    df_val["completion"].to_numpy(dtype="float32"), CONFIG.y_transform
)
y_test = transformed_prices(
    df_test["completion"].to_numpy(dtype="float32"), CONFIG.y_transform
)

print("Target ranges (log-transformed):")
print(f"  Train: [{y_train.min():.2f}, {y_train.max():.2f}]")
print(f"  Val: [{y_val.min():.2f}, {y_val.max():.2f}]")
print(f"  Test: [{y_test.min():.2f}, {y_test.max():.2f}]")

## Optuna Objective Function

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """Optuna objective optimizing learning rate, dropout, architecture, and weight decay.

    Returns best validation MAE in dollars (lower is better).
    """
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    dropout_base = trial.suggest_float("dropout_base", 0.05, 0.3)
    initial_dim = 2 ** trial.suggest_int("initial_dim_power", 7, 10)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-1, log=True)

    print(f"\n{'=' * 80}")
    print(f"Trial {trial.number}: Testing hyperparameters:")
    print(f"  Learning Rate: {learning_rate:.2e}")
    print(f"  Dropout Base: {dropout_base:.3f}")
    print(f"  Initial Dim: {initial_dim}")
    print(f"  Weight Decay: {weight_decay:.2e}")
    print(f"{'=' * 80}\n")

    input_dim = X_train.shape[1] * X_train.shape[2]
    model = CompressionPriceNet(
        dropout_base=dropout_base,
        initial_dim=initial_dim,
        input_dim=input_dim,
    ).to(device)

    train_loader = DataLoader(
        PriceDataset(X_train, y_train),
        batch_size=CONFIG.batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        PriceDataset(X_val, y_val),
        batch_size=CONFIG.batch_size,
        shuffle=False,
    )

    criterion = nn.MSELoss()
    optimizer = optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=learning_rate,
        epochs=CONFIG.epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        anneal_strategy="cos",
    )

    early_stop_fn = create_early_stopping_fn(
        patience=10,
        loss_patience=3,
        loss_threshold=1.5,
        warmup_epochs=10,
    )
    best_val_mae = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        convert_back_fn=convert_back_y,
        epochs=CONFIG.epochs,
        grad_clip=CONFIG.grad_clip,
        trial=trial,
        early_stopping_fn=early_stop_fn,
    )

    print(f"\nTrial {trial.number} completed with best Val MAE: ${best_val_mae:.2f}\n")

    return best_val_mae

## Run Optuna Study

In [ ]:
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=10,
    ),
    study_name="compression_price_net_optimization",
)

print(f"\n{'=' * 80}")
print(f"Starting Optuna study with {CONFIG.n_trials} trials")
print(f"{'=' * 80}\n")

study.optimize(objective, n_trials=CONFIG.n_trials, show_progress_bar=True)

print(f"\n{'=' * 80}")
print("Optimization completed!")
print(f"{'=' * 80}\n")

## Results Analysis

In [ ]:
print("\n" + "=" * 80)
print("TOP 5 TRIALS")
print("=" * 80)

top_trials = sorted(
    study.trials, key=lambda t: t.value if t.value is not None else float("inf")
)[:5]

for rank, trial in enumerate(top_trials, 1):
    print(f"\n{'─' * 80}")
    print(f"Rank {rank}: Trial #{trial.number}")
    print(f"{'─' * 80}")
    print(f"Validation MAE: ${trial.value:.2f}")
    print(f"\nHyperparameters:")

    params = trial.params
    lr = params["learning_rate"]
    wd = params["weight_decay"]
    db = params["dropout_base"]
    dim_power = params["initial_dim_power"]
    dim = 2**dim_power

    print(f"  learning_rate:  {lr:.2e}")
    print(f"  weight_decay:   {wd:.2e}")
    print(
        f"  dropout_base:   {db:.3f} (dropout1={min(2 * db, 0.5):.3f}, dropout2={db:.3f})"
    )
    print(f"  initial_dim:    {dim} (2^{dim_power})")

print("\n" + "=" * 80)
print(f"\nBest Trial: #{study.best_trial.number} with Val MAE: ${study.best_value:.2f}")
print("=" * 80)

In [ ]:
print("\n" + "=" * 80)
print("PARAMETER IMPORTANCE")
print("=" * 80 + "\n")

importance = optuna.importance.get_param_importances(study)
for param, imp in importance.items():
    print(f"  {param}: {imp:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig = optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title("Optimization History: Validation MAE over Trials")
plt.ylabel("Best Validation MAE ($)")
plt.xlabel("Trial Number")
plt.tight_layout()
plt.show()

In [ ]:
fig = optuna.visualization.matplotlib.plot_param_importances(study)
plt.title("Hyperparameter Importances")
plt.tight_layout()
plt.show()

In [ ]:
fig = optuna.visualization.matplotlib.plot_parallel_coordinate(study)
plt.title("Parallel Coordinate Plot")
plt.tight_layout()
plt.show()

## Train Final Model with Best Parameters

In [ ]:
print("\n" + "=" * 80)
print("TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS")
print("=" * 80 + "\n")

best_params = study.best_params
learning_rate = best_params["learning_rate"]
dropout_base = best_params["dropout_base"]
initial_dim = 2 ** best_params["initial_dim_power"]
weight_decay = best_params["weight_decay"]

# Load full training data
df_train = load_data_from_hf(split="train")

input_dim = X_train.shape[1] * X_train.shape[2]
final_model = CompressionPriceNet(
    dropout_base=dropout_base,
    initial_dim=initial_dim,
    input_dim=input_dim,
).to(device)

train_loader = DataLoader(
    PriceDataset(X_train, y_train),
    batch_size=CONFIG.batch_size,
    shuffle=True,
)
val_loader = DataLoader(
    PriceDataset(X_val, y_val),
    batch_size=CONFIG.batch_size,
    shuffle=False,
)
test_loader = DataLoader(
    PriceDataset(X_test, y_test),
    batch_size=CONFIG.batch_size,
    shuffle=False,
)

criterion = nn.MSELoss()
optimizer = optim.AdamW(
    final_model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=learning_rate,
    epochs=CONFIG.epochs,
    steps_per_epoch=len(train_loader),
    pct_start=0.1,
    anneal_strategy="cos",
)
early_stop_fn = create_early_stopping_fn(
    patience=10,
    loss_patience=3,
    loss_threshold=1.5,
    warmup_epochs=10,
)

best_val_mae = train_model(
    model=final_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    convert_back_fn=convert_back_y,
    epochs=CONFIG.epochs,
    grad_clip=CONFIG.grad_clip,
    trial=None,  # No pruning for final model
    early_stopping_fn=early_stop_fn,
)

print(f"\nFinal model achieved Val MAE: ${best_val_mae:.2f}")

## Evaluate on Test Set

In [ ]:
print("\n" + "=" * 80)
print("FINAL TEST EVALUATION")
print("=" * 80 + "\n")

test_metrics = evaluate_model(
    final_model, test_loader, criterion, device, convert_back_y
)

print("Test Set Results:")
print(f"  Test Loss: {test_metrics['loss']:.4f}")
print(f"  Test MAE: ${test_metrics['mae']:.2f}")

print("\n" + "=" * 80)

## Save Results

In [ ]:
import json
from datetime import datetime

results = {
    "study_name": study.study_name,
    "n_trials": len(study.trials),
    "best_trial_number": study.best_trial.number,
    "best_val_mae": study.best_value,
    "best_params": study.best_params,
    "test_mae": test_metrics["mae"],
    "test_loss": test_metrics["loss"],
    "timestamp": datetime.now().isoformat(),
    "config": {
        "data_percent": CONFIG.data_percent,
        "y_transform": CONFIG.y_transform,
        "batch_size": CONFIG.batch_size,
        "epochs": CONFIG.epochs,
        "grad_clip": CONFIG.grad_clip,
        "cache_path": str(CONFIG.cache_path),
        "cache_filename_root": CONFIG.cache_filename_root,
        "n_trials": CONFIG.n_trials,
        "seed": CONFIG.seed,
    },
}

results_path = Path(
    "/content/drive/MyDrive/Pricer_Embeddings/optuna_results_compression_net.json"
)
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {results_path}")

# Save final model
model_path = Path("/content/drive/MyDrive/Pricer_Embeddings/best_compression_net.pth")
torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "best_params": study.best_params,
        "val_mae": best_val_mae,
        "test_mae": test_metrics["mae"],
        "config": {
            "data_percent": CONFIG.data_percent,
            "y_transform": CONFIG.y_transform,
            "batch_size": CONFIG.batch_size,
            "epochs": CONFIG.epochs,
            "grad_clip": CONFIG.grad_clip,
            "cache_path": str(CONFIG.cache_path),
            "cache_filename_root": CONFIG.cache_filename_root,
            "n_trials": CONFIG.n_trials,
            "seed": CONFIG.seed,
        },
    },
    model_path,
)

print(f"Model saved to {model_path}")